#### Name : Prathmesh Nitnaware

#### Roll no.: 23102B0060

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, StackingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

In [ ]:
DATA_PATH = r"./SMSSpamCollection"

df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['label', 'message'])

df['label'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['message']
y = df['label']

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5572, 2)


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [26]:
def evaluate_model(model, X, y, k=5):

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    precision_list = []
    recall_list = []
    f1_list = []
    roc_list = []

    for train_idx, test_idx in skf.split(X, y):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        precision_list.append(precision_score(y_test, preds))
        recall_list.append(recall_score(y_test, preds))
        f1_list.append(f1_score(y_test, preds))

        try:
            probs = model.predict_proba(X_test)[:, 1]
            roc_list.append(roc_auc_score(y_test, probs))
        except:
            roc_list.append(np.nan)

    return {
        "Precision": np.mean(precision_list),
        "Recall": np.mean(recall_list),
        "F1": np.mean(f1_list),
        "ROC_AUC": np.nanmean(roc_list)
    }

In [15]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2))

In [16]:
print("\n===== Naive Bayes =====")

nb_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", MultinomialNB())
])

evaluate_model(nb_pipeline, X, y)


===== Naive Bayes =====
Precision: 1.0000 ± 0.0000
Recall:    0.7189 ± 0.0192
F1-Score:  0.8363 ± 0.0130
ROC-AUC:   0.9850 ± 0.0050

Confusion Matrix:
[[4825    0]
 [ 210  537]]


In [17]:
print("\n===== Logistic Regression =====")

lr_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(lr_pipeline, X, y)


===== Logistic Regression =====
Precision: 0.9863 ± 0.0098
Recall:    0.6560 ± 0.0191
F1-Score:  0.7876 ± 0.0113
ROC-AUC:   0.9907 ± 0.0058

Confusion Matrix:
[[4818    7]
 [ 257  490]]


In [18]:
print("\n===== Linear SVM =====")

svm_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", SVC(kernel='linear', probability=True))
])

evaluate_model(svm_pipeline, X, y)


===== Linear SVM =====
Precision: 0.9718 ± 0.0131
Recall:    0.9157 ± 0.0206
F1-Score:  0.9427 ± 0.0110
ROC-AUC:   0.9935 ± 0.0043

Confusion Matrix:
[[4805   20]
 [  63  684]]


In [21]:
print("\n===== Hard Voting =====")

voting_hard = VotingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    voting='hard'
)

pipeline = Pipeline([("tfidf", tfidf), ("clf", voting_hard)])

evaluate_model(pipeline, X, y)


===== Hard Voting =====
Precision: 0.9884 ± 0.0085
Recall:    0.7965 ± 0.0082
F1-Score:  0.8821 ± 0.0081
ROC-AUC:   Not Available for Hard Voting

Confusion Matrix:
[[4818    7]
 [ 152  595]]


In [22]:
print("\n===== Soft Voting =====")

voting_soft = VotingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    voting='soft'
)

pipeline = Pipeline([("tfidf", tfidf), ("clf", voting_soft)])

evaluate_model(pipeline, X, y)


===== Soft Voting =====
Precision: 0.9850 ± 0.0068
Recall:    0.8835 ± 0.0180
F1-Score:  0.9314 ± 0.0122
ROC-AUC:   0.9926 ± 0.0048

Confusion Matrix:
[[4815   10]
 [  87  660]]


In [23]:
print("\n===== Stacking =====")

stacking = StackingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    final_estimator=LogisticRegression()
)

pipeline = Pipeline([("tfidf", tfidf), ("clf", stacking)])

evaluate_model(pipeline, X, y)


===== Stacking =====
Precision: 0.9529 ± 0.0070
Recall:    0.9478 ± 0.0183
F1-Score:  0.9502 ± 0.0101
ROC-AUC:   0.9927 ± 0.0044

Confusion Matrix:
[[4790   35]
 [  39  708]]


In [24]:
print("\n===== AdaBoost (Decision Stumps) =====")

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100
)

pipeline = Pipeline([("tfidf", tfidf), ("clf", ada)])

evaluate_model(pipeline, X, y)


===== AdaBoost (Decision Stumps) =====
Precision: 0.9550 ± 0.0140
Recall:    0.4284 ± 0.0247
F1-Score:  0.5912 ± 0.0252
ROC-AUC:   0.9282 ± 0.0133

Confusion Matrix:
[[4810   15]
 [ 427  320]]


In [29]:
best_model = StackingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    final_estimator=LogisticRegression()
)

final_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", best_model)
])

In [30]:
from sklearn.ensemble import StackingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

best_model = StackingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    final_estimator=LogisticRegression()
)

final_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", best_model)
])

In [31]:
results = {}

results["Naive Bayes"] = evaluate_model(nb_pipeline, X, y)
results["Logistic Regression"] = evaluate_model(lr_pipeline, X, y)
results["SVM"] = evaluate_model(svm_pipeline, X, y)
results["Stacking"] = evaluate_model(final_pipeline, X, y)

In [32]:
comparison_df = pd.DataFrame(results).T
comparison_df.reset_index(inplace=True)
comparison_df.rename(columns={"index": "Model"}, inplace=True)

comparison_df.to_csv("ensemble_comparison.csv", index=False)
print("Saved: ensemble_comparison.csv")

Saved: ensemble_comparison.csv


In [33]:
comparison_df.to_csv("ensemble_comparison.csv", index=False)
print("\nSaved: ensemble_comparison.csv")


best_model = StackingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    final_estimator=LogisticRegression()
)

final_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", best_model)
])

final_pipeline.fit(X, y)

predictions = final_pipeline.predict(X)
probabilities = final_pipeline.predict_proba(X)[:, 1]

final_predictions_df = pd.DataFrame({
    "MessageId": np.arange(len(X)),
    "Actual": y,
    "Predicted": predictions,
    "Probability": probabilities
})

final_predictions_df.to_csv("final_model_predictions.csv", index=False)
print("Saved: final_model_predictions.csv")


Saved: ensemble_comparison.csv
Saved: final_model_predictions.csv
